# EDA

In [2]:
import pandas as pd
import ast

path = "../data/raw" 
products_df = pd.read_csv(f"{path}/product_info.csv")
print(products_df.columns)

df_reviews_1 = pd.read_csv(f"{path}/reviews_0-250.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_2 = pd.read_csv(f"{path}/reviews_250-500.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_3 = pd.read_csv(f"{path}/reviews_500-750.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_4 = pd.read_csv(f"{path}/reviews_750-1250.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)
df_reviews_5 = pd.read_csv(f"{path}/reviews_1250-end.csv", index_col=0, dtype={'author_id': 'str'}, low_memory=False)

df_reviews = pd.concat([df_reviews_1,df_reviews_2,df_reviews_3,df_reviews_4,df_reviews_5],axis=0)
print(df_reviews.columns)

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')
Index(['author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')


In [3]:
class DataProcessing:

    columns_to_drop = ['variation_desc', 'sale_price_usd', 'value_price_usd', 'child_max_price', 'child_min_price',            
                       'helpfulness','review_title','tertiary_category','highlights','variation_value',            
                       'variation_type', 'size','total_feedback_count','total_neg_feedback_count',   
                       'total_pos_feedback_count','submission_time', 'limited_edition',            
                       'sephora_exclusive','child_count', 'brand_id', 'online_only', 'new', 
                       'out_of_stock', 'reviews']

    critical_columns = ['ingredients', 'review_text', 'author_id', 'rating', 
                        'product_name', 'brand_name', 'price_usd','secondary_category']
    
    user_attribute_columns = ['skin_tone', 'eye_color', 'skin_type', 'hair_color']

    water_keywords = ['water', 'aqua', 'hydrosol']

    silicone_keywords = ['cyclopentasiloxane', 'cyclohexasiloxane', 'dimethicone',
                         'trimethicone', 'amodimethicone', 'vinyl dimethicone', 
                         'cetyl dimethicone', 'phenyl trimethicone','silicone']

    def __init__(self, products_df, reviews_df):
        self.products_df = products_df
        self.reviews_df = reviews_df

    
    def cols_to_use(self):
        """
        Returns the columns to use from the products_df and reviews_df DataFrames
        """
        cols_to_use = self.products_df.columns.difference(self.reviews_df.columns)
        cols_to_use = list(cols_to_use)
        cols_to_use.append('product_id')
        return cols_to_use
    

    def merge_dataframes(self):
        """
        Merges the products_df and reviews_df DataFrames on the 'product_id' column and drop unnecessary columns
        """
        cols_to_use = self.cols_to_use()
        self.merged_df = pd.merge(self.reviews_df, self.products_df[cols_to_use], how='outer', on=['product_id', 'product_id'])
        self.merged_df.drop(columns=self.columns_to_drop, inplace=True, axis=1)
    

    def nan_handler(self):
        """
        Fills NaN values in the DataFrame with empty strings
        """
        self.merged_df.dropna(subset=self.critical_columns, how='any', inplace=True)

        for col in self.user_attribute_columns:
            self.merged_df[col].fillna('Unknown', inplace=True)
        
        self.merged_df.drop(columns=['primary_category'], axis=1, inplace=True)
    
    
    def ingredients_to_string(self, ingredients_list):
        """
        Convert a string representation of a list into a single string.
        
        Parameters:
            val (str): A string that represents a list, e.g., "['Water, Butylene Glycol, ...']".
            
        Returns:
            str: A single string with all the list items joined by a space.
        """
        try:
            items = ast.literal_eval(ingredients_list)

            if isinstance(items, list):
                return " ".join(items)
            else:
                return ingredients_list
            
        except Exception as e:
            return ingredients_list
    

    def apply_processing(self):
        """
        Applies the processing functions to the specified column of the DataFrame.
        """
        self.merged_df['ingredients_cleaned'] = self.merged_df['ingredients'].apply(self.ingredients_to_string)


    def water_or_silicone(self):
        """
        Creates two new columns in the DataFrame, one for water-based products and one for silicone-based products.
        return 2 boolean columns
        """
        self.merged_df["water_based"] = self.merged_df['ingredients'].apply(
            lambda ingredients: any(keyword in ingredients.lower() for keyword in self.water_keywords)
        )

        self.merged_df['silicone_based'] = self.merged_df['ingredients'].apply(
            lambda ingredients: any(keyword in ingredients.lower() for keyword in self.silicone_keywords)
        )

In [4]:
data_processer = DataProcessing(products_df, df_reviews)

In [5]:
data_processer.merge_dataframes()
data_processer.nan_handler()
data_processer.apply_processing()
data_processer.water_or_silicone()

merged_df = data_processer.merged_df
merged_df.columns

/tmp/ipykernel_50332/4059936692.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.merged_df[col].fillna('Unknown', inplace=True)


Index(['author_id', 'rating', 'is_recommended', 'review_text', 'skin_tone',
       'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name',
       'brand_name', 'price_usd', 'ingredients', 'loves_count',
       'secondary_category', 'ingredients_cleaned', 'water_based',
       'silicone_based'],
      dtype='object')

In [6]:
merged_df.sample(5)

,author_id,rating,is_recommended,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,ingredients,loves_count,secondary_category,ingredients_cleaned,water_based,silicone_based
1793,1173475766,5.0,NaN,"If you have problem skin, this could be a grea...",fair,Unknown,combination,Unknown,P114902,Goodbye Acne Max Complexion Correction Pads,Peter Thomas Roth,48.0,"['Salicylic Acid 2%, Alcohol Denat., Water/Aqu...",56955,Treatments,"Salicylic Acid 2%, Alcohol Denat., Water/Aqua/...",True,False
236440,2776710623,4.0,1.0,I use both the coconut and the rose - I think ...,mediumTan,brown,combination,black,P409800,Cleansing & Exfoliating Wipes,SEPHORA COLLECTION,3.0,"['Water, Caprylic/Capric Triglyceride, Glyceri...",266116,Cleansers,"Water, Caprylic/Capric Triglyceride, Glycerin,...",True,False
492184,1552685071,5.0,1.0,I am 43 years of age. I have dry skin and I a...,mediumTan,brown,dry,brown,P436094,Evercalm Overnight Recovery Balm,REN Clean Skincare,55.0,"['Coco-Caprylate/Caprate, Glycerin, Aqua (Wate...",38294,Moisturizers,"Coco-Caprylate/Caprate, Glycerin, Aqua (Water)...",True,False
976391,27691828776,4.0,1.0,This product was great! It left my skin feelin...,lightMedium,brown,combination,brown,P481164,Essential Energy Hydrating Day Cream Broad Spe...,Shiseido,50.0,"['Purpose Avobenzone 2.0%, Sunscreen Homosalat...",2628,Moisturizers,"Purpose Avobenzone 2.0%, Sunscreen Homosalate ...",True,True
1097090,5110531656,5.0,1.0,This product has me hooked on the Caudalie bra...,light,hazel,dry,blonde,P94421,Vinoperfect Radiance Dark Spot Serum Vitamin C...,Caudalie,82.0,"['Aqua/Water/Eau, Butylene Glycol, Glycerin, C...",166423,Treatments,"Aqua/Water/Eau, Butylene Glycol, Glycerin, Coc...",True,False


In [7]:
merged_df['column_to_vectorize'] = merged_df[['skin_tone', 'eye_color', 'skin_type', 'hair_color', 
                                              'ingredients_cleaned', 'water_based', 'silicone_based']].values.tolist()

merged_df.sample(5)

,author_id,rating,is_recommended,review_text,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,ingredients,loves_count,secondary_category,ingredients_cleaned,water_based,silicone_based,column_to_vectorize
268264,5594663836,4.0,1.0,This product goes on smoothly and makes my ski...,medium,brown,combination,black,P415619,Dreamskin Skin Perfector,Dior,130.00,"['Aqua (Water), Dimethicone, Cyclopentasiloxan...",40209,Moisturizers,"Aqua (Water), Dimethicone, Cyclopentasiloxane,...",True,True,"[medium, brown, combination, black, Aqua (Wate..."
196344,2044627185,4.0,NaN,I got this in my June play box. And like most ...,Unknown,Unknown,Unknown,Unknown,P396623,Makeup Setting Spray Organic Sunscreen SPF 30,COOLA,36.00,"['Avobenzone 2.8%, Homosalate 3.5%, Octisalate...",34125,Sunscreen,"Avobenzone 2.8%, Homosalate 3.5%, Octisalate 3...",True,False,"[Unknown, Unknown, Unknown, Unknown, Avobenzon..."
616681,5543615944,3.0,0.0,I do like how the application is very cold fee...,mediumTan,brown,combination,black,P448802,Brighten-i Eye Cream,The INKEY List,12.99,"['Water, dimethicone, coco-caprylate/caprate, ...",87730,Eye Care,"Water, dimethicone, coco-caprylate/caprate, gl...",True,True,"[mediumTan, brown, combination, black, Water, ..."
969900,2067001413,1.0,0.0,"I wanted to like this, but I do not have sensi...",lightMedium,blue,combination,brown,P480630,Mini Jet Lag Mask,Summer Fridays,26.00,"['Water/Aqua/Eau, Caprylic/Capric Triglyceride...",2900,Masks,"Water/Aqua/Eau, Caprylic/Capric Triglyceride, ...",True,False,"[lightMedium, blue, combination, brown, Water/..."
1040376,8643286619,5.0,1.0,Definitely recommend this scrub!! Some scrubs ...,lightMedium,blue,combination,auburn,P501188,SATOCANE Pore Purifying Scrub Mask,WASO,38.00,"['Water(Aqua/Eau),Kaolin,Dipropylene Glycol,Gl...",1673,Masks,"Water(Aqua/Eau),Kaolin,Dipropylene Glycol,Glyc...",True,False,"[lightMedium, blue, combination, auburn, Water..."
